### Ingest Platform Users CSV
This notebook ingests the platform users file from the landing layer, standardizes the dataset, and writes it to a bronze Delta table.

Business context:
* This table shows the people who registered on the platform
* Each row represents a user profile with contact, account, billing, and shipping details
* The bronze table preserves these records for downstream analysis of user activity, customer coverage, and platform operations

#### Step 1: Load shared setup values
This step imports the common project configuration used across bronze ingestion notebooks.

What happens here:
* Loads shared Unity Catalog and volume variables
* Reuses helper functions such as metadata column enrichment
* Keeps environment-specific values centralized in one setup notebook


#### Step 2: Build the platform users file path
This step assembles the full storage path for the raw platform users CSV.

What happens here:
* Combines the configured volume, catalog, schema, and folder names
* Points to the specific dated platform users file to ingest
* Makes the resolved source path easy to validate before reading data


#### Step 3: Define the platform users schema
This step declares the expected structure of the CSV so Spark reads user, contact, address, and account fields with the intended data types.

What happens here:
* Defines the incoming platform user columns explicitly
* Applies consistent types for user identifiers and bid limit fields
* Avoids schema drift that can happen with CSV inference alone


#### Platform users data dictionary
This file represents people who registered on the platform, where each row shows a user account together with contact, company, billing, shipping, and account status details.

Key fields:
* `User #`: the platform user identifier
* `First Name`, `Last Name`, and `Email Address`: the primary identity and contact details for the user
* `Company Name` and `Company Url`: company-related information associated with the account
* `Items Won` and `Highest Won`: summary indicators of user auction outcomes
* Billing and shipping fields: address and fulfillment-related details stored on the user account
* `Status`: the current account or user status
* `Bid Limit`: the configured bidding limit for the user account
* `Date Created`: when the user record was created in the platform

Business use:
* Track who has registered on the platform
* Support customer reporting and account management analysis
* Provide a base user dataset for downstream auction, fulfillment, and engagement workflows


#### Step 4: Load and enrich the platform users data
This step reads the raw CSV into Spark and appends standard metadata columns used for traceability.

What happens here:
* Loads the platform users file into a Spark DataFrame
* Applies the predefined schema during ingestion
* Adds metadata such as source path and ingestion timestamp


#### Step 5: Prepare the bronze-ready users dataset
This step standardizes the DataFrame so the columns are easier to use after the bronze write.

What happens here:
* Renames source columns into a lowercase, underscore-based format
* Cleans spaces and user profile field names for downstream usability
* Confirms the bronze target context before persisting the table


#### Step 6: Write the bronze platform users table
This final step persists the cleaned platform users DataFrame as a Delta table in the bronze schema.

What happens here:
* Writes the transformed platform user data to a managed Delta table
* Rebuilds the target table on reruns using overwrite mode
* Makes the bronze dataset available for downstream silver transformations, customer analysis, and operational reporting


In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Build the full source path for the CSV file to ingest from cloud storage.
# This keeps the input file location parameterized through shared environment values.
path = f'/{volume}/{catalog}/{schema}/{file}/{batch_id}/item description.csv'

In [ ]:
df_item_desc = spark.read.format('csv') \
    .option('header', 'true') \
         .option('multiLine', 'true') \
    .option("quote", '"') \
    .option("escape", '"') \
        .option('timestampFormat', 'yyyy-MM-dd HH:mm:ss z') \
    .option('inferSchema', 'True') \
    .load(path)
    # .load('abfss://auction@company.dfs.core.windows.net/item description.csv')

In [ ]:
df_item_desc = df_item_desc.toDF(*[column_name.strip() for column_name in df_item_desc.columns])

In [ ]:
df_item_desc = df_item_desc.withColumns({'lot_number':F.col('lot_number').cast('int'), 'quantity':F.col('quantity').cast('int'),'item_name':F.col('item_name').cast('string')})

In [ ]:
df_item_desc = add_columns(df_item_desc)

In [ ]:
# Preview the target bronze namespace components before writing the Delta table.
f'{volume}/{catalog}/{bronze}'

In [ ]:
# Write the ingested DataFrame to the bronze layer as a Delta table.
# Overwrite mode is used so the table can be rebuilt on repeated runs.
df_item_desc \
    .write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('batch_id') \
    .option('replaceWhere',f"batch_id = '{batch_id}'")\
    .saveAsTable(f'{catalog}.{bronze}.item_desc')